# Decoder-Only Transformer: Genre-Conditioned Story Generation

This notebook implements a **Decoder-Only Transformer** from scratch in PyTorch for genre-conditioned story generation. It is fully self-contained and covers:

1. **Data Loading & Preprocessing Integration** (using our `src` modules).
2. **Tokenizer & Vocabulary setup** (Whitespace tokenizer with special tokens).
3. **Pipeline Visualization & Alignment** (visualizing how input sequences $X$ map to target sequences $Y$).
4. **Decoder-Only Transformer Architecture from Scratch** with detailed math and LaTeX formulas.
5. **Interactive Diagnostic Shape Verification** for each individual block of the model.
6. **Training and Validation loops** utilizing automated checkpointing and device detection via `utils.py`.
7. **Autoregressive Generation** with Temperature Scaling and Top-K Filtering.
8. **Attention Map Visualization** to inspect attention heads during generation.

---

### Pipeline Overview

A decoder-only transformer works auto-regressively. During generation:
```
Prompt (e.g. "<GENRE_Horror> the house was") 
  --> Tokenizer 
  --> Input IDs 
  --> Embedding + Positional Encoding 
  --> Multi-Head Attention (with Causal Masking) 
  --> Feed Forward 
  --> Linear Head 
  --> Softmax + Temperature/Top-K 
  --> Next Token
```
This is repeated to generate the story word-by-word.

## 1. Imports, Reproducibility, & Device Detection

We set random seeds for reproducibility and use the project's utility module to identify the best training device (using GPU/MPS if available, fallback to CPU).

In [ ]:
import math
import random
import sys
from collections import Counter
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Add src directory to system path
sys.path.append("../src")
from utils import get_device, load_checkpoint, print_model_summary, save_checkpoint, set_seed

# Set seed for reproducibility
set_seed(42)

# Get best available device (automatically detects Apple Silicon MPS, CUDA, or CPU)
DEVICE = get_device()
print("Device:", DEVICE)


## 2. Load Dataset Splits

We load the train, validation, and test datasets created in the Preprocessing step.

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

print("train / val / test sizes:", len(train_df), len(val_df), len(test_df))


## 3. Genre-Conditioning & Text Preparation

To condition the model on a specific genre, we prepend a special genre tag to the story: `<GENRE_{genre}>`. This teaches the transformer to associate the genre token with the theme and style of the text that follows.

Stories are kept at full length (up to **1000 words**, matching the preprocessing notebook) before tokenization.


In [ ]:
def make_text(row):
    genre = str(row["genre"]).replace(" ", "_")
    return f"<GENRE_{genre}> " + str(row["story"])

train_texts = train_df.apply(make_text, axis=1).tolist()
val_texts = val_df.apply(make_text, axis=1).tolist()
test_texts = test_df.apply(make_text, axis=1).tolist()

# quick sanity check
print("Example:", train_texts[0][:200])


## 4. Tokenizer & Vocabulary

We implement a simple whitespace tokenizer. The vocabulary contains the special tokens:
- `<PAD>`: Used to pad shorter sequences to the maximum sequence length.
- `<UNK>`: Represents words not found in the training vocabulary.
- `<SOS>`: Start of sequence token.
- `<EOS>`: End of sequence token.

In [ ]:
SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
counter = Counter()
for t in train_texts:
    counter.update(t.lower().split())

vocab = SPECIAL_TOKENS + sorted(counter.keys())
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)

print("Vocab size:", VOCAB_SIZE)


In [ ]:
def encode(text: str) -> List[int]:
    return [word2idx.get(w, word2idx["<UNK>"]) for w in text.lower().split()]

MAX_WORDS = 1000          # matches preprocessing notebook
MAX_LEN = 1024            # genre tag + 1000 words + SOS/EOS
BATCH_SIZE = 8            # reduced from 32; lower to 4 if OOM on MPS/CUDA

def prepare_sequence(text: str, max_len: int = MAX_LEN) -> List[int]:
    seq = [word2idx["<SOS>"]]
    seq += encode(text)
    seq += [word2idx["<EOS>"]]
    seq = seq[:max_len]
    seq += [word2idx["<PAD>"]] * (max_len - len(seq))
    return seq


## 5. Input-Target Alignment Visualization

A key concept in auto-regressive training is that the model predicts the *next* token at each step.
Thus, the target sequence $Y$ is the input sequence $X$ shifted to the left by one position.
Let's visualize this alignment to build absolute intuition:

In [ ]:
# Create a toy example to visualize input-target alignment
sample_text = "once upon a time"
encoded_seq = prepare_sequence(sample_text, max_len=10)
input_tokens = [idx2word[i] for i in encoded_seq[:-1]]
target_tokens = [idx2word[i] for i in encoded_seq[1:]]

print(f"{'Input (X)':<25} -> {'Target (Y)':<25}")
print("-" * 55)
for inp, tar in zip(input_tokens, target_tokens):
    print(f"{inp:<25} -> {tar:<25}")


## 6. Encode Dataset Splits to Fixed-Length Sequences

We convert all the text datasets to numpy integer matrices of shape `(num_samples, MAX_LEN)` where `MAX_LEN=1024` fits full 1000-word stories plus genre tag and special tokens.


In [ ]:
train_sequences = [prepare_sequence(t) for t in train_texts]
val_sequences = [prepare_sequence(t) for t in val_texts]
test_sequences = [prepare_sequence(t) for t in test_texts]

train_sequences = np.array(train_sequences, dtype=np.int64)
val_sequences = np.array(val_sequences, dtype=np.int64)
test_sequences = np.array(test_sequences, dtype=np.int64)

print("train shape:", train_sequences.shape)
print("val shape:", val_sequences.shape)
print("test shape:", test_sequences.shape)

non_pad = (train_sequences != word2idx["<PAD>"]).sum(axis=1)
print(f"Token counts — mean: {non_pad.mean():.0f}, max: {non_pad.max()}")


## 7. Dataset & PyTorch Dataloaders

We create a custom PyTorch `Dataset` that slices the inputs `x = seq[:-1]` and shifted targets `y = seq[1:]`.

In [ ]:
class StoryDataset(Dataset):
    def __init__(self, sequences: np.ndarray):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        x = seq[:-1]  # input tokens (seq_len - 1)
        y = seq[1:]   # target tokens shifted by 1 (seq_len - 1)
        return x, y

train_loader = DataLoader(StoryDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(StoryDataset(val_sequences), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(StoryDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False)

print("Dataloaders ready")


## 8. Model Architecture Implementation & Diagnostics

Here we implement the Transformer architecture from scratch in PyTorch. For each component, we include:
- A brief explanation.
- LaTeX formula showing the math.
- A **Diagnostic Block** to verify the expected input/output tensor shapes.

### 8.1 Positional Encoding

Since the self-attention mechanism does not process tokens sequentially (it processes all tokens in parallel), it has no inherent sense of token order. We inject positional information by adding a unique vector to the embedding of each token, calculated using sine and cosine functions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)].to(x.device)
        return x

# Diagnostic Verification
toy_pe = PositionalEncoding(d_model=256)
toy_input = torch.zeros(4, 50, 256)  # batch=4, seq_len=50, d_model=256
toy_output = toy_pe(toy_input)
print(f"Positional Encoding shapes:")
print(f"  Input Shape:  {toy_input.shape}")
print(f"  Output Shape: {toy_output.shape} (Expected: [4, 50, 256])")
assert toy_output.shape == toy_input.shape


### 8.2 Causal (Subsequent) Mask

In generative text generation, we must prevent the model from looking at future tokens when predicting the current token.
A causal mask is a upper-triangular matrix of boolean values. Let's see what a causal mask looks like:

In [ ]:
def subsequent_mask(size):
    # Mask out subsequent positions (1 = allowed, 0 = masked)
    attn_shape = (1, size, size)
    subsequent = torch.triu(torch.ones(attn_shape), diagonal=1).bool()
    return ~subsequent

# Diagnostic visualization of subsequent mask for sequence length of 5
toy_mask = subsequent_mask(5)
print("Causal Mask for seq_len=5:")
print(toy_mask[0].int())


### 8.3 Multi-Headed Self-Attention

Attention queries $Q$, keys $K$, and values $V$ are computed. Scaled dot-product attention computes compatibility:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

With Multi-Headed Self-Attention, we run this in parallel across $H$ heads, letting the model pay attention to different parts of the input sequence.

In [ ]:
class MultiHeadedSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linear_q = nn.Linear(d_model, d_model)
        self.linear_k = nn.Linear(d_model, d_model)
        self.linear_v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        batch_size = x.size(0)
        q = self.linear_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        k = self.linear_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        v = self.linear_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            mask = mask.unsqueeze(1).to(torch.bool)
            scores = scores.masked_fill(~mask, float('-inf'))
        probs = torch.softmax(scores, dim=-1)
        probs = self.dropout(probs)
        context = torch.matmul(probs, v)
        context = context.transpose(1,2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.out(context)

# Diagnostic Verification
toy_attn = MultiHeadedSelfAttention(d_model=256, num_heads=8)
toy_input = torch.zeros(4, 50, 256)
toy_mask = subsequent_mask(50).to(torch.bool).expand(4, 50, 50)  # batch size expansion
toy_output = toy_attn(toy_input, mask=toy_mask)
print(f"Multi-Headed Self-Attention shapes:")
print(f"  Input Shape:  {toy_input.shape}")
print(f"  Mask Shape:   {toy_mask.shape}")
print(f"  Output Shape: {toy_output.shape} (Expected: [4, 50, 256])")
assert toy_output.shape == toy_input.shape


### 8.4 FeedForward Network

The FeedForward block consists of two linear transformations with a ReLU activation in between:

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

# Diagnostic Verification
toy_ff = FeedForward(d_model=256, d_ff=1024)
toy_input = torch.zeros(4, 50, 256)
toy_output = toy_ff(toy_input)
print(f"FeedForward Network shapes:")
print(f"  Input Shape:  {toy_input.shape}")
print(f"  Output Shape: {toy_output.shape} (Expected: [4, 50, 256])")
assert toy_output.shape == toy_input.shape


### 8.5 Decoder Layer

A single Decoder Layer consists of a Multi-Head Self-Attention sub-layer and a FeedForward sub-layer. We apply residual connections and Layer Normalization after each sub-layer.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadedSelfAttention(d_model, num_heads, dropout=dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        sa = self.self_attn(x, mask=mask)
        x = x + self.dropout(sa)
        x = self.norm1(x)
        ff = self.feed_forward(x)
        x = x + self.dropout(ff)
        x = self.norm2(x)
        return x

# Diagnostic Verification
toy_layer = DecoderLayer(d_model=256, num_heads=8, d_ff=1024)
toy_input = torch.zeros(4, 50, 256)
toy_mask = subsequent_mask(50).expand(4, 50, 50)
toy_output = toy_layer(toy_input, mask=toy_mask)
print(f"Decoder Layer shapes:")
print(f"  Input Shape:  {toy_input.shape}")
print(f"  Output Shape: {toy_output.shape} (Expected: [4, 50, 256])")
assert toy_output.shape == toy_input.shape


### 8.6 Decoder-Only Transformer Model

The final model stacks $N$ Decoder Layers, maps the final outputs through a LayerNorm, and projects them to the vocabulary dimension to output logits for each word.

In [ ]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=6, d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout=dropout) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    def forward(self, x, mask=None):
        x = self.token_emb(x) * math.sqrt(self.token_emb.embedding_dim)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask=mask)
        x = self.ln_f(x)
        return self.head(x)  # logits

# Diagnostic Verification
toy_transformer = DecoderOnlyTransformer(vocab_size=VOCAB_SIZE, num_layers=4)
toy_input = torch.randint(0, VOCAB_SIZE, (4, 50))  # batch=4, seq_len=50
toy_mask = subsequent_mask(50).expand(4, 50, 50)
toy_output = toy_transformer(toy_input, mask=toy_mask)
print(f"Decoder-Only Transformer shapes:")
print(f"  Input Tokens Shape: {toy_input.shape}")
print(f"  Output Logits Shape: {toy_output.shape} (Expected: [4, 50, {VOCAB_SIZE}])")
assert toy_output.shape == (4, 50, VOCAB_SIZE)


## 9. Instantiating Model & Optimizer

We create our model and define cross entropy loss (ignoring the `<PAD>` token index so padding does not affect gradients) and the optimizer. We print out a summary of model parameters using our utilities.

In [ ]:
model = DecoderOnlyTransformer(
    VOCAB_SIZE, d_model=256, num_heads=8, num_layers=4,
    d_ff=1024, max_len=MAX_LEN, dropout=0.1
).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Print clean summary using helper
print_model_summary(model)


## 10. Training & Validation Loop

We execute the training loop over 10 epochs. We track losses and save checkpoints using the project's utility modules.

In [ ]:
from tqdm.notebook import tqdm
EPOCHS = 10
best_val_loss = float('inf')
train_losses = []
val_losses = []
checkpoint_dir = Path('../checkpoints')
checkpoint_dir.mkdir(exist_ok=True, parents=True)

def make_causal_mask(batch_size, seq_len):
    mask = subsequent_mask(seq_len).to(DEVICE)
    return mask.expand(batch_size, seq_len, seq_len)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            mask = make_causal_mask(x.size(0), x.size(1))
            logits = model(x, mask=mask)
            loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1))
            total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    return avg_loss, math.exp(avg_loss)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} train')
    for x, y in pbar:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad()
        mask = make_causal_mask(x.size(0), x.size(1))
        logits = model(x, mask=mask)
        loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    avg_train = total_loss / len(train_loader)
    train_losses.append(avg_train)
    avg_val, val_ppl = evaluate(model, val_loader)
    val_losses.append(avg_val)
    print(f'Epoch {epoch}: train_loss={avg_train:.4f} val_loss={avg_val:.4f} val_ppl={val_ppl:.2f}')
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        ckpt_path = checkpoint_dir / 'best_story_transformer.pt'
        save_checkpoint(model, optimizer, epoch, avg_val, ckpt_path, val_loss=avg_val, vocab_size=VOCAB_SIZE)

# save final checkpoint
save_checkpoint(model, optimizer, EPOCHS, val_losses[-1], checkpoint_dir / 'last_story_transformer.pt', val_loss=val_losses[-1], vocab_size=VOCAB_SIZE)
print('Training complete')


## 11. Test Set Evaluation

Evaluate the best checkpoint on the held-out test split (never seen during training or validation).


In [ ]:
# Load best checkpoint before final test eval
best_ckpt = checkpoint_dir / 'best_story_transformer.pt'
if best_ckpt.exists():
    load_checkpoint(str(best_ckpt), model)

test_loss, test_ppl = evaluate(model, test_loader)
print(f"Test loss: {test_loss:.4f}  |  Test perplexity: {test_ppl:.2f}")


## 12. Plot Training & Validation Loss

We visualize training and validation loss trends over epochs.


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label='train', marker='o')
plt.plot(val_losses, label='val', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()


## 13. Autoregressive Sampling & Generation

We implement autoregressive generation. We can scale logits by a **temperature** parameter (higher temperature increases randomness) and apply **top-k** sampling (limiting choices to the top-k most likely tokens) to generate fluent stories.


In [ ]:
def top_k_logits(logits, k):
    if k == 0:
        return logits
    v, ix = torch.topk(logits, k)
    out = logits.clone()
    out[out < v[..., -1, None]] = -float('Inf')
    return out

def sample_sequence(model, prompt: str, max_tokens=100, temperature=1.0, top_k=0):
    model.eval()
    tokens = prepare_sequence(prompt)
    tokens = tokens[:MAX_LEN]
    cur = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    generated = []
    with torch.no_grad():
        for _ in range(max_tokens):
            mask = subsequent_mask(cur.size(1)).to(DEVICE).expand(cur.size(0), -1, -1)
            logits = model(cur, mask=mask)[0, -1]  # Get last token logits
            logits = logits / max(temperature, 1e-8)
            if top_k > 0:
                logits = top_k_logits(logits, top_k)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_id)
            if next_id == word2idx['<EOS>']:
                break
            cur = torch.cat([cur, torch.tensor([[next_id]], device=DEVICE)], dim=1)
    return ' '.join([idx2word.get(i, '<UNK>') for i in generated])

prompt = '<GENRE_Horror> the house was silent'
sample_out = sample_sequence(model, prompt, max_tokens=50, temperature=1.0, top_k=40)
print("Prompt: ", prompt)
print("Generated Story continuation:")
print(sample_out)


## 14. Visualizing Attention Weights (Self-Attention Head Map)

Let's see what the transformer is looking at! We can extract attention maps from the model layers to inspect how the generated tokens attend back to prior prompt tokens, like the `<GENRE_...>` tag.


In [ ]:
def get_attention_weights(model, text: str):
    """Passes a single sequence through the model to retrieve self-attention probabilities
    from the first attention layer.
    """
    model.eval()
    tokens = prepare_sequence(text)
    cur = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    # We manually simulate the first layer to extract the attention weights.
    with torch.no_grad():
        # Embedding + positional encoding
        x = model.token_emb(cur) * math.sqrt(model.token_emb.embedding_dim)
        x = model.pos_enc(x)
        
        # Retrieve first layer attention module
        attn_module = model.layers[0].self_attn
        
        # Calculate Query, Key, Value
        batch_size = x.size(0)
        q = attn_module.linear_q(x).view(batch_size, -1, attn_module.num_heads, attn_module.d_k).transpose(1,2)
        k = attn_module.linear_k(x).view(batch_size, -1, attn_module.num_heads, attn_module.d_k).transpose(1,2)
        
        scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(attn_module.d_k)
        mask = subsequent_mask(x.size(1)).to(DEVICE)
        mask = mask.unsqueeze(1).to(torch.bool)
        scores = scores.masked_fill(~mask, float('-inf'))
        probs = torch.softmax(scores, dim=-1)
        
        # probs shape: [batch_size, num_heads, seq_len, seq_len]
        return probs[0].cpu().numpy(), [idx2word[i] for i in tokens]

# Let's plot the attention weights of Layer 0, Head 0 for a short prompt
probs, tokens = get_attention_weights(model, "<GENRE_Mystery> the footprints led")

# We plot a sub-matrix of size 8x8 for clarity
seq_len_to_show = min(12, len(tokens))
plt.figure(figsize=(10, 8))
sns.heatmap(
    probs[0, :seq_len_to_show, :seq_len_to_show],
    xticklabels=tokens[:seq_len_to_show],
    yticklabels=tokens[:seq_len_to_show],
    annot=True,
    cmap="Blues",
    fmt=".2f"
)
plt.title("Self-Attention Map (Layer 0, Head 0)")
plt.xlabel("Attended Token")
plt.ylabel("Query Token")
plt.show()
